# SentryVision — Development Notebook

This notebook documents the **exploration path** that led to the production pipeline: baseline detection, adding tracking, adding face recognition, and finally the integrated pipeline with business logic (attendance/alerts).

**The maintained, production version of this logic lives in [`src/sentryvision/`](../src/sentryvision/) and is used by [`app.py`](../app.py).** This notebook is kept for portfolio/reference purposes — showing the step-by-step reasoning — and is not imported by the app.


## Setup — Install Dependencies

In [ ]:
# Clean uninstall first, then let pip resolve mutually-compatible versions
# on its own instead of forcing a specific numpy version. Colab's pre-installed
# packages (torch, scipy, etc.) are built against numpy 2.x, so pinning numpy
# back to 1.x creates ABI mismatches with those compiled extensions.

!pip install -q opencv-python-headless ultralytics face_recognition

# IMPORTANT: after this cell finishes, go to Runtime -> Restart session,
# then run the rest of the notebook from the top. A restart is required so
# that the freshly (re)installed packages are loaded fresh, instead of an
# older, already-imported version staying cached in this session's memory.


## Step 1 — Baseline Person Detection

Single-model inference on a video file: run YOLOv8 frame-by-frame, filter for the `person` class, and save annotated sample frames.

This is a reference baseline only — every frame's detections are independent, so the same person in frame 30 and frame 31 are treated as unrelated boxes.

In [ ]:
import os
import cv2
from google.colab import files
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# COCO class 0 = 'person' -> filtering only person detections
VIDEO_PATH = '/content/sample_video.mp4'

if not os.path.exists(VIDEO_PATH):
    print("No existing video found. Please upload one.")
    uploaded = files.upload()
    uploaded_filename = list(uploaded.keys())[0]
    os.rename(uploaded_filename, VIDEO_PATH)
else:
    print(f"Reusing existing video: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise IOError("Failed to open video file. Check the file path.")
print(f"Video loaded: {VIDEO_PATH}")

OUTPUT_DIR = '/content/outputs_baseline'
os.makedirs(OUTPUT_DIR, exist_ok=True)

frame_count = 0
saved_count = 0
SAVE_INTERVAL = 30  # save every Nth frame to avoid excessive disk writes

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, classes=[0], conf=0.5, iou=0.45, verbose=False)
    annotated_frame = results[0].plot()

    frame_count += 1
    if frame_count % SAVE_INTERVAL == 0:
        output_path = os.path.join(OUTPUT_DIR, f'frame_{frame_count:05d}.jpg')
        cv2.imwrite(output_path, annotated_frame)
        saved_count += 1

cap.release()
print(f"Processing complete. Total frames processed: {frame_count}, saved: {saved_count}")


## Step 2 — Add Object Tracking (ByteTrack)

`model.predict()` treats every frame independently. `model.track()` with `persist=True` assigns a persistent ID to each person across frames using ByteTrack — the foundation needed before recognition results can be cached per person instead of re-run on every single frame.

In [ ]:
model = YOLO('yolov8n.pt')

cap = cv2.VideoCapture(VIDEO_PATH)
OUTPUT_DIR = '/content/outputs_tracked'
os.makedirs(OUTPUT_DIR, exist_ok=True)

frame_count = 0
saved_count = 0
unique_track_ids = set()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # persist=True tells the tracker to carry track state across frames
    results = model.track(
        frame, classes=[0], conf=0.5, iou=0.45,
        persist=True, tracker="bytetrack.yaml", verbose=False
    )
    annotated_frame = results[0].plot()

    if results[0].boxes.id is not None:
        unique_track_ids.update(results[0].boxes.id.int().tolist())

    frame_count += 1
    if frame_count % SAVE_INTERVAL == 0:
        cv2.imwrite(os.path.join(OUTPUT_DIR, f'frame_{frame_count:05d}.jpg'), annotated_frame)
        saved_count += 1

cap.release()
print(f"Processing complete. Total frames: {frame_count}, saved: {saved_count}")
print(f"Total unique persons tracked in video: {len(unique_track_ids)}")


## Step 3 — Face Recognition

Build a database of known-face encodings, then match a test image against it.

**Note on `MIN_FACE_AREA_RATIO`:** during testing, small non-face patterns (e.g. a circular logo on a t-shirt) were occasionally detected as faces by the underlying detector. Filtering out detections below a minimum area ratio relative to the crop removes these without needing a second model.

In [ ]:
import face_recognition

KNOWN_FACES_DIR = '/content/known_faces'

known_encodings = []
known_names = []

for filename in os.listdir(KNOWN_FACES_DIR):
    filepath = os.path.join(KNOWN_FACES_DIR, filename)
    image = face_recognition.load_image_file(filepath)
    encodings = face_recognition.face_encodings(image)

    if len(encodings) == 0:
        print(f"Warning: No face detected in {filename}. Skipping.")
        continue

    known_encodings.append(encodings[0])
    known_names.append(os.path.splitext(filename)[0])

print(f"Loaded {len(known_names)} known face(s): {known_names}")


In [ ]:
from google.colab import files
import numpy as np

MIN_FACE_AREA_RATIO = 0.002  # empirically tuned: filters logos/small artifacts while keeping real faces
FACE_MATCH_TOLERANCE = 0.5

print("Upload a test image to check face recognition:")
uploaded = files.upload()
test_image_path = list(uploaded.keys())[0]

test_image = face_recognition.load_image_file(test_image_path)
raw_face_locations = face_recognition.face_locations(test_image)

image_height, image_width = test_image.shape[:2]
image_area = image_height * image_width

filtered_locations = []
for (top, right, bottom, left) in raw_face_locations:
    face_area = (right - left) * (bottom - top)
    area_ratio = face_area / image_area
    if area_ratio >= MIN_FACE_AREA_RATIO:
        filtered_locations.append((top, right, bottom, left))
    else:
        print(f"Filtered out small detection: area ratio = {area_ratio:.4f}")

print(f"Faces before filtering: {len(raw_face_locations)}")
print(f"Faces after filtering: {len(filtered_locations)}")

test_encodings = face_recognition.face_encodings(test_image, filtered_locations)

if len(test_encodings) == 0:
    print("No valid face detected after filtering.")
else:
    for i, encoding in enumerate(test_encodings):
        matches = face_recognition.compare_faces(known_encodings, encoding, tolerance=FACE_MATCH_TOLERANCE)
        face_distances = face_recognition.face_distance(known_encodings, encoding)
        if True in matches and len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            name = known_names[best_match_index]
            confidence = 1 - face_distances[best_match_index]
            print(f"Face {i+1}: Match found - {name} (confidence: {confidence:.2f})")
        else:
            print(f"Face {i+1}: Unknown person")


## Step 4 — Integrated Pipeline (Tracking + Recognition + Business Logic)

This cell was the final exploratory version that combined tracking, recognition, and SQLite-backed attendance/alert logging in one script. **This logic has since been refactored into [`src/sentryvision/pipeline.py`](../src/sentryvision/pipeline.py)** — see that module (via `app.py`) for the version actually used by the dashboard.

Kept here only to show the dev-time reasoning: track IDs are cached once resolved (`track_id_to_name`), and dedup sets (`logged_names`, `alerted_track_ids`) ensure each business-logic event fires exactly once.

## Appendix — Debugging Aids

Two small utilities used during development to sanity-check the pipeline's output, kept here for reference.

In [ ]:
# Inspect every face crop that was fed into the recognizer, labeled by
# track ID and frame number — useful for checking whether a track_id -> name
# mapping makes sense (e.g. two different track IDs both matching the same
# known person suggests the tracker lost and re-acquired them).
import os
from IPython.display import Image, display

DEBUG_DIR = '/content/debug_face_crops'
if not os.path.exists(DEBUG_DIR):
    print(f"Directory does not exist: {DEBUG_DIR}")
else:
    debug_files = sorted(os.listdir(DEBUG_DIR))
    print(f"Total files found in {DEBUG_DIR}: {len(debug_files)}")
    for f in debug_files[:10]:
        display(Image(os.path.join(DEBUG_DIR, f)))


In [ ]:
# Read back everything written to attendance_log and security_alerts
# during a pipeline run.
import sqlite3

conn = sqlite3.connect('/content/sentryvision.db')
cursor = conn.cursor()

print("=== Attendance Log ===")
for row in cursor.execute("SELECT id, name, track_id, entry_time FROM attendance_log ORDER BY id"):
    print(row)

print("\n=== Security Alerts ===")
for row in cursor.execute("SELECT id, track_id, detected_time, snapshot_path FROM security_alerts ORDER BY id"):
    print(row)

conn.close()
